In [119]:
from pyspark.sql import SparkSession, functions as F
spark=(
      SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")  
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.shuffle.partitions", 500) 
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.autoBroadcastJoinThreshold", 52428800)
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/work/postgresql-42.7.3.jar") ##test
     .config('spark.sql.session.timeZone', 'Asia/Dubai') 
    .getOrCreate()  
)
hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [120]:
jdbc_url = 'jdbc:postgresql://postgres:5432/lakehouse'
jdbc_props = {
    'user':     'matrix',
    'password': 'matrix123',
    'driver':   'org.postgresql.Driver'
}

In [121]:
import os
file_path = "/home/jovyan/work"
files = os.listdir(file_path)
print(files)



['docker-compose.yaml', 'spark-defaults.conf', 'notebooks', 'output.txt', 'Untitled(7)(3)(1).ipynb', 'spark1.ipynb', 'spark1.bash', 'card_trn.csv', 'cust.csv', 'spark2.ipynb', 'task2_transacion.csv', 'part1', 'part2', 'spark3.ipynb', 'init-scripts', 'pg-backups', 'spark4.ipynb', 'spark4.bash', 'postgresql-42.7.3.jar', '.vscode', 'spark4.sql', 'practice.ipynb', 'mcc_mapping.csv', 'fx_rate.csv', 'customers.csv', 'cards.csv', '02_load.sql', '01_ddl.sql', 'landing']


In [122]:
bronze_path = "s3a://bronze/"
silver_path = "s3a://silver/"
gold_path = "s3a://gold/"

In [123]:
# df_mcc_mapping = spark.read.format("csv").option("header", "true").load("/home/jovyan/work/mcc_mapping.csv")
# df_fx_rate=spark.read.format("csv").option("header","true").load("/home/jovyan/work/fx_rate.csv")
# df_customers=spark.read.format("csv").option("header","true").load("/home/jovyan/work/customers.csv")
# df_cards=spark.read.format("csv").option("header","true").load("/home/jovyan/work/cards.csv")

In [124]:
df_atm_yesterday = spark.read.format("csv").option("header", "true").load("/home/jovyan/work/landing/atm/tx_date=2026-02-20/atm_tx.csv")
df_atm_yesterday=df_atm_yesterday.withColumn("type", F.lit("atm"))
df_ecom_yesterday = spark.read.format("csv").option("header", "true").load("/home/jovyan/work/landing/ecom/tx_date=2026-02-20/ecom_tx.csv")
df_ecom_yesterday=df_ecom_yesterday.withColumn("type", F.lit("ecom"))



In [125]:
df_atm_yesterday.show(5)
df_ecom_yesterday.show(5)


+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|type|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+
|TX0698572469|2026-02-20 13:05:00|CARD300077|6011|  -5.0|     GBP|    AZE|    Baku|   ATM65006| atm|
|TX3189450479|2026-02-20 05:30:00|CARD300147|6011|673.05|     TRY|    AZE|   Ganja|   ATM14262| atm|
|TX7971451951|2026-02-20 22:57:00|CARD300144|6011|724.37|     AZN|    DEU|  Berlin|   ATM62840| atm|
|TX2113712998|2026-02-20 09:37:00|CARD300081|6011|650.26|     TRY|    USA|New York|   ATM35075| atm|
|TX8523830552|2026-02-20 17:13:00|CARD300021|6011|477.99|     GBP|    AZE|   Ganja|   ATM08832| atm|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+
only showing top 5 rows

+------------+-------------------+----------+-----------+----+----

In [126]:
df_ecom_yesterday.show(5)

+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
|       tx_id|              tx_ts|   card_no|customer_no| mcc|amount|currency|country|    city|type|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
|        NULL|2026-02-20 03:05:00|CARD300094| CUST100073|4829|481.32|     TRY|   AZE |   Ganja|ecom|
|TX2009750632|2026-02-20 10:04:00|CARD300065|       NULL|7995|440.94|     GBP|    USA|   Miami|ecom|
|TX4203529010|2026-02-20 13:00:00|CARD300044| CUST100037|4111| 20.66|     AZN|    GEO| Tbilisi|ecom|
|TX0229410673|2026-02-20 14:40:00|CARD300148| CUST100114|6011| 49.44|     AZN|   DEU |  Berlin|ecom|
|TX9974847299|2026-02-20 18:51:00|CARD300077|       NULL|7011|  7.67|     USD|    TUR|Istanbul|ecom|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
only showing top 5 rows



In [127]:
df_ecom_yesterday.show(5)

+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
|       tx_id|              tx_ts|   card_no|customer_no| mcc|amount|currency|country|    city|type|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
|        NULL|2026-02-20 03:05:00|CARD300094| CUST100073|4829|481.32|     TRY|   AZE |   Ganja|ecom|
|TX2009750632|2026-02-20 10:04:00|CARD300065|       NULL|7995|440.94|     GBP|    USA|   Miami|ecom|
|TX4203529010|2026-02-20 13:00:00|CARD300044| CUST100037|4111| 20.66|     AZN|    GEO| Tbilisi|ecom|
|TX0229410673|2026-02-20 14:40:00|CARD300148| CUST100114|6011| 49.44|     AZN|   DEU |  Berlin|ecom|
|TX9974847299|2026-02-20 18:51:00|CARD300077|       NULL|7011|  7.67|     USD|    TUR|Istanbul|ecom|
+------------+-------------------+----------+-----------+----+------+--------+-------+--------+----+
only showing top 5 rows



In [128]:
df_atm_yesterday.write.mode("overwrite").csv(bronze_path+"atm/tx_date=2026-02-20/atm_tx.csv", header=True)

In [129]:
df_ecom_yesterday.write.mode("overwrite").csv(bronze_path+"ecom/tx_date=2026-02-20/ecom_tx.csv", header=True)

In [130]:
df_pos_yesterday=spark.read.json("/home/jovyan/work/landing/pos/tx_date=2026-02-20/pos_events.json")


In [131]:
df_pos_yesterday.show(5)

+------+----------+--------+-------------------+--------------------+----+------------+
|amount|   card_no|currency|         event_time|            location| mcc|       tx_id|
+------+----------+--------+-------------------+--------------------+----+------------+
|109.43|CARD300026|     AZN|2026-02-20 15:11:00|{Sumqayit, AZE, T...|5411|        NULL|
|145.87|CARD300069|     TRY|2026-02-20 12:21:00|{Sumqayit, AZE, T...|7011|TX3989232788|
| 23.43|CARD300083|     USD|2026-02-20 10:23:00|{Baku, AZE, T832108}|6011|TX3673638609|
| 54.54|CARD300114|     AZN|2026-02-20 01:47:00|{Miami, USA, T629...|6300|TX2422797582|
|103.31|CARD300155|     AZN|2026-02-20 12:13:00|{Berlin, DEU, T83...|5812|TX6964079195|
+------+----------+--------+-------------------+--------------------+----+------------+
only showing top 5 rows



In [132]:
df_pos_yesterday.printSchema()

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- location: struct (nullable = true)
 |    |-- city: string (nullable = true)
 |    |-- country: string (nullable = true)
 |    |-- terminal_id: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable = true)



In [133]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType, DoubleType, StringType, BooleanType
schema_for_pos = StructType([
	StructField("amount", StringType(), True),
	StructField("card_no", StringType(), True),
	StructField("currency", StringType(), True),
	StructField("event_time", DateType(), True),
	StructField("location", StructType([
		StructField("city", StringType(), True),
		StructField("country", StringType(), True),
		StructField("terminal_id", StringType(), True)
	]), True),
	StructField("mcc", StringType(), True),
	StructField("tx_id", StringType(), True)
])
    
    


In [134]:
df_pos_yesterday = spark.read.format("json").schema(schema_for_pos).load("/home/jovyan/work/landing/pos/tx_date=2026-02-20/pos_events.json")

In [135]:
df_pos_yesterday=df_pos_yesterday.withColumn("city",F.col("location.city"))
df_pos_yesterday=df_pos_yesterday.withColumn("country",F.col("location.country"))
df_pos_yesterday=df_pos_yesterday.withColumn("terminal_id",F.col("location.terminal_id"))
df_pos_yesterday=df_pos_yesterday.withColumn("type", F.lit("pos"))
df_pos_yesterday=df_pos_yesterday.drop("location")

In [136]:
df_ecom_yesterday.show()

+------------+-------------------+----------+-----------+----+-------+--------+-------+--------+----+
|       tx_id|              tx_ts|   card_no|customer_no| mcc| amount|currency|country|    city|type|
+------------+-------------------+----------+-----------+----+-------+--------+-------+--------+----+
|        NULL|2026-02-20 03:05:00|CARD300094| CUST100073|4829| 481.32|     TRY|   AZE |   Ganja|ecom|
|TX2009750632|2026-02-20 10:04:00|CARD300065|       NULL|7995| 440.94|     GBP|    USA|   Miami|ecom|
|TX4203529010|2026-02-20 13:00:00|CARD300044| CUST100037|4111|  20.66|     AZN|    GEO| Tbilisi|ecom|
|TX0229410673|2026-02-20 14:40:00|CARD300148| CUST100114|6011|  49.44|     AZN|   DEU |  Berlin|ecom|
|TX9974847299|2026-02-20 18:51:00|CARD300077|       NULL|7011|   7.67|     USD|    TUR|Istanbul|ecom|
|TX4587867306|2026-02-20 01:27:00|CARD300155|       NULL|4829| 744.15|     GBP|    DEU|  Munich|ecom|
|TX5030637492|2026-02-20 01:20:00|CARD300006|       NULL|6011|  17.00|     GBP|   

In [137]:
df_pos_yesterday.show(5)

+------+----------+--------+----------+----+------------+--------+-------+-----------+----+
|amount|   card_no|currency|event_time| mcc|       tx_id|    city|country|terminal_id|type|
+------+----------+--------+----------+----+------------+--------+-------+-----------+----+
|109.43|CARD300026|     AZN|2026-02-20|5411|        NULL|Sumqayit|    AZE|    T885318| pos|
|145.87|CARD300069|     TRY|2026-02-20|7011|TX3989232788|Sumqayit|    AZE|    T105155| pos|
| 23.43|CARD300083|     USD|2026-02-20|6011|TX3673638609|    Baku|    AZE|    T832108| pos|
| 54.54|CARD300114|     AZN|2026-02-20|6300|TX2422797582|   Miami|    USA|    T629051| pos|
|103.31|CARD300155|     AZN|2026-02-20|5812|TX6964079195|  Berlin|    DEU|    T838337| pos|
+------+----------+--------+----------+----+------------+--------+-------+-----------+----+
only showing top 5 rows



In [138]:
from pyspark.sql import DataFrame


In [139]:
df_atm_yesterday.printSchema()
df_ecom_yesterday.printSchema()
df_pos_yesterday.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- type: string (nullable = false)

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- customer_no: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- type: string (nullable = false)

root
 |-- amount: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- event_time: date (nullable = true)
 |-- mcc: string (nullable = true)
 |-- tx_id: string (nullable 

In [140]:
final_df = df_atm_yesterday.unionByName(df_ecom_yesterday, allowMissingColumns=True).unionByName(df_pos_yesterday, allowMissingColumns=True)


In [141]:
final_df.select("type").distinct().show()

+----+
|type|
+----+
| atm|
|ecom|
| pos|
+----+



In [142]:
from pyspark.sql.functions import lit
final_df=final_df.withColumn("insert_time",F.current_timestamp())

In [143]:
final_df.show()

+------------+-------------------+----------+----+------+--------+-------+----------+-----------+----+-----------+----------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|      city|terminal_id|type|customer_no|event_time|         insert_time|
+------------+-------------------+----------+----+------+--------+-------+----------+-----------+----+-----------+----------+--------------------+
|TX0698572469|2026-02-20 13:05:00|CARD300077|6011|  -5.0|     GBP|    AZE|      Baku|   ATM65006| atm|       NULL|      NULL|2026-02-24 06:33:...|
|TX3189450479|2026-02-20 05:30:00|CARD300147|6011|673.05|     TRY|    AZE|     Ganja|   ATM14262| atm|       NULL|      NULL|2026-02-24 06:33:...|
|TX7971451951|2026-02-20 22:57:00|CARD300144|6011|724.37|     AZN|    DEU|    Berlin|   ATM62840| atm|       NULL|      NULL|2026-02-24 06:33:...|
|TX2113712998|2026-02-20 09:37:00|CARD300081|6011|650.26|     TRY|    USA|  New York|   ATM35075| atm|       NULL|    

In [144]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, StringType, DateType
final_df=final_df.withColumn("amount", col("amount").cast(IntegerType()))
final_df=final_df.withColumn("mcc", col("mcc").cast(IntegerType()))




In [145]:
##final_df.show()
final_df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- type: string (nullable = false)
 |-- customer_no: string (nullable = true)
 |-- event_time: date (nullable = true)
 |-- insert_time: timestamp (nullable = false)



In [146]:
final_df.orderBy("tx_id").show()

+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+-----------+----------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|type|customer_no|event_time|         insert_time|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+-----------+----------+--------------------+
|        NULL|2026-02-20 03:05:00|CARD300094|4829|   481|     TRY|   AZE |   Ganja|       NULL|ecom| CUST100073|      NULL|2026-02-24 06:33:...|
|        NULL|               NULL|CARD300026|5411|   109|     AZN|    AZE|Sumqayit|    T885318| pos|       NULL|2026-02-20|2026-02-24 06:33:...|
|TX0008465429|2026-02-20 02:32:00|CARD300116|4829|  1082|     EUR|    USA|New York|       NULL|ecom| CUST100090|      NULL|2026-02-24 06:33:...|
|TX0010439369|2026-02-20 21:23:00|CARD300058|5732|   127|     GBP|    AZE|Sumqayit|       NULL|ecom| CUST100047|      NULL|2026-02

In [147]:
final_df=final_df.distinct()

In [148]:
final_df.groupBy('tx_id').count().show()

+------------+-----+
|       tx_id|count|
+------------+-----+
|TX1105846844|    1|
|TX3369375471|    1|
|TX2520448691|    1|
|TX6283096313|    1|
|TX4548972509|    1|
|TX8928603242|    1|
|TX7738607823|    1|
|TX2096742753|    1|
|TX8292028710|    1|
|TX1153807899|    1|
|TX2535096487|    1|
|TX8157853801|    1|
|TX6640102630|    1|
|TX1704697467|    1|
|TX1177031938|    1|
|TX3514352716|    1|
|TX4723043838|    1|
|TX8980230727|    1|
|TX4412778594|    1|
|TX4480130205|    1|
+------------+-----+
only showing top 20 rows



In [149]:
from pyspark.sql.window import Window

final_df = final_df.withColumn(
    "row_num", 
    F.row_number().over(Window.partitionBy("tx_id").orderBy(F.col("insert_time").desc()))
).filter(F.col("row_num") == 1).drop("row_num")

In [150]:
final_df.write.mode("overwrite").partitionBy("insert_time").parquet(bronze_path+"transactions/")

In [151]:
final_df.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- type: string (nullable = false)
 |-- customer_no: string (nullable = true)
 |-- event_time: date (nullable = true)
 |-- insert_time: timestamp (nullable = false)



In [152]:
final_df.createOrReplaceTempView("final_df")
##final_df=spark.sql("SELECT * FROM final_df where tx_id is not null and tx_ts is not null and amount>0 and currency is not null")
final_df=spark.sql("SELECT *, case when tx_id is null then 'tx_id is null' when tx_ts is null then 'tx_ts is null' when amount<=0 then 'amount is less than or equal to 0' when currency is null then 'currency is null' else 'valid' end as validation_status FROM final_df")



In [153]:
final_df.filter(F.col("validation_status") != "valid").show()


+------------+-------------------+----------+----+------+--------+-------+----------+-----------+----+-----------+----------+--------------------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|      city|terminal_id|type|customer_no|event_time|         insert_time|   validation_status|
+------------+-------------------+----------+----+------+--------+-------+----------+-----------+----+-----------+----------+--------------------+--------------------+
|        NULL|2026-02-20 03:05:00|CARD300094|4829|   481|     TRY|   AZE |     Ganja|       NULL|ecom| CUST100073|      NULL|2026-02-24 06:33:...|       tx_id is null|
|TX0013801510|               NULL|CARD300139|6300|    44|     AZN|    GEO|   Tbilisi|    T347608| pos|       NULL|2026-02-20|2026-02-24 06:33:...|       tx_ts is null|
|TX0062646657|               NULL|CARD300027|7995|  1162|     AZN|    DEU|    Berlin|    T367255| pos|       NULL|2026-02-20|2026-02-24 06:33:...|       tx_ts i

In [154]:
final_good=spark.sql("SELECT * FROM final_df where tx_id is not null and tx_ts is not null and amount>0 and currency is not null")
final_bad=spark.sql("SELECT * FROM final_df where tx_id is null or tx_ts is null or amount<=0 or currency is null")

In [155]:
transaction_good=bronze_path = "s3a://bronze/transactions_good"
transaction_bad=bronze_path = "s3a://bronze/transactions_bad"

In [156]:
final_good.write.mode("overwrite").parquet(silver_path+"transactions_good") 

In [157]:
final_bad.write.mode("overwrite").parquet(transaction_bad)
final_good.write.mode("overwrite").parquet(transaction_good)

In [158]:

df_mcc_mapping = spark.read.format("csv").option("header", "true").load("/home/jovyan/work/mcc_mapping.csv")
df_fx_rate=spark.read.format("csv").option("header","true").load("/home/jovyan/work/fx_rate.csv")
df_customers=spark.read.format("csv").option("header","true").load("/home/jovyan/work/customers.csv")
df_cards=spark.read.format("csv").option("header","true").load("/home/jovyan/work/cards.csv")

In [159]:
df_mcc_mapping.write.mode("overwrite").format("delta").save("s3a://bronze/dims/mcc_mapping")
df_fx_rate.write.mode("overwrite").format("delta").save("s3a://bronze/dims/fx_rate")
df_customers.write.mode("overwrite").format("delta").save("s3a://bronze/dims/customers")
df_cards.write.mode("overwrite").format("delta").save("s3a://bronze/dims/cards")

In [166]:
final_good.createOrReplaceTempView("final_good")
final_good = spark.sql("""
    SELECT 
        tx_id,
        tx_ts,
        card_no,
        mcc,
        amount,
        upper(trim(currency)) as currency,
        upper(trim(country))  as country,
        city,
        terminal_id,
        type,
        customer_no,
        event_time,
        insert_time
    FROM final_good
""")

final_good.show(5)




+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+-----------+----------+--------------------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|type|customer_no|event_time|         insert_time|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+-----------+----------+--------------------+
|TX0008465429|2026-02-20 02:32:00|CARD300116|4829|  1082|     EUR|    USA|New York|       NULL|ecom| CUST100090|      NULL|2026-02-24 06:34:...|
|TX0010439369|2026-02-20 21:23:00|CARD300058|5732|   127|     GBP|    AZE|Sumqayit|       NULL|ecom| CUST100047|      NULL|2026-02-24 06:34:...|
|TX0052654111|2026-02-20 16:33:00|CARD300113|5411|    88|     GBP|    AZE|Sumqayit|       NULL|ecom| CUST100088|      NULL|2026-02-24 06:34:...|
|TX0054410614|2026-02-20 12:48:00|CARD300058|6011|   302|     EUR|    GBR|  London|   ATM04564| atm|       NULL|      NULL|2026-02

In [161]:
# docker cp /home/nurgun/Documents/feb10sinif/customers.csv postgres:/tmp/customers.csv
# docker cp /home/nurgun/Documents/feb10sinif/cards.csv     postgres:/tmp/cards.csv
# docker cp /home/nurgun/Documents/feb10sinif/mcc_mapping.csv postgres:/tmp/mcc_mapping.csv
# docker cp /home/nurgun/Documents/feb10sinif/fx_rate.csv   postgres:/tmp/fx_rate.csv

In [162]:
# docker exec -it postgres psql -U matrix -d lakehouse -c "COPY public.customers(customer_no, full_name, home_country) FROM '/tmp/customers.csv' CSV HEADER;"
# docker exec -it postgres psql -U matrix -d lakehouse -c "COPY public.cards(card_no, customer_no) FROM '/tmp/cards.csv' CSV HEADER;"
# docker exec -it postgres psql -U matrix -d lakehouse -c "COPY public.mcc_mapping(mcc, category, risk_weight) FROM '/tmp/mcc_mapping.csv' CSV HEADER;"
# docker exec -it postgres psql -U matrix -d lakehouse -c "COPY public.fx_rate(fx_date, currency, rate_to_azn) FROM '/tmp/fx_rate.csv' CSV HEADER;"

In [167]:
final_good.printSchema()

root
 |-- tx_id: string (nullable = true)
 |-- tx_ts: string (nullable = true)
 |-- card_no: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: integer (nullable = true)
 |-- currency: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- terminal_id: string (nullable = true)
 |-- type: string (nullable = false)
 |-- customer_no: string (nullable = true)
 |-- event_time: date (nullable = true)
 |-- insert_time: timestamp (nullable = false)



In [168]:

final_good.createOrReplaceTempView("final_good")
df_cards.createOrReplaceTempView("cards")

final_good_enriched = spark.sql("""
    SELECT 
        t.*,
        COALESCE(t.customer_no, c.customer_no) AS customer_no_filled
    FROM final_good t
    LEFT JOIN cards c ON t.card_no = c.card_no
""")

final_good_enriched = final_good_enriched.drop("customer_no") \
    .withColumnRenamed("customer_no_filled", "customer_no")

final_good_enriched.show(5)

+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+
|TX0008465429|2026-02-20 02:32:00|CARD300116|4829|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100090|
|TX0010439369|2026-02-20 21:23:00|CARD300058|5732|   127|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100047|
|TX0052654111|2026-02-20 16:33:00|CARD300113|5411|    88|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100088|
|TX0054410614|2026-02-20 12:48:00|CARD300058|6011|   302|     EUR|    GBR|  London|   ATM04564| atm|      NULL|2026-02-24 06:34:..

In [169]:

df_customers.createOrReplaceTempView("customers")
final_good_enriched.createOrReplaceTempView("final_good_enriched")

final_good_enriched = spark.sql("""
    SELECT 
        t.*,
        c.home_country,
        CASE WHEN t.country != c.home_country THEN true ELSE false END AS is_foreign
    FROM final_good_enriched t
    LEFT JOIN customers c ON t.customer_no = c.customer_no
""")

final_good_enriched.show(5)

+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+
|       tx_id|              tx_ts|   card_no| mcc|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|home_country|is_foreign|
+------------+-------------------+----------+----+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+
|TX0008465429|2026-02-20 02:32:00|CARD300116|4829|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100090|         AZE|      true|
|TX0010439369|2026-02-20 21:23:00|CARD300058|5732|   127|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100047|         GEO|      true|
|TX0052654111|2026-02-20 16:33:00|CARD300113|5411|    88|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06:34:...| CUST100088|         TU

In [171]:
df_mcc_mapping.createOrReplaceTempView("mcc_mapping")
final_good_enriched.createOrReplaceTempView("final_good_enriched")

from pyspark.sql.functions import broadcast

final_good_enriched = final_good_enriched.join(
    broadcast(df_mcc_mapping.select("mcc", "category", "risk_weight")),
    on="mcc",
    how="left"
)

final_good_enriched.show(5)

+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+
| mcc|       tx_id|              tx_ts|   card_no|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|home_country|is_foreign|      category|risk_weight|
+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+
|4829|TX0008465429|2026-02-20 02:32:00|CARD300116|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:39:...| CUST100090|         AZE|      true|Money Transfer|          8|
|5732|TX0010439369|2026-02-20 21:23:00|CARD300058|   127|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06:39:...| CUST100047|         GEO|      true|   Electronics|          4|
|5411|TX0052654111|2

In [172]:
df_fx_rate.createOrReplaceTempView("fx_rate")
final_good_enriched.createOrReplaceTempView("final_good_enriched")

final_good_enriched = spark.sql("""
    SELECT 
        t.*,
        f.rate_to_azn,
        round(t.amount * f.rate_to_azn, 2) AS amount_azn
    FROM final_good_enriched t
    LEFT JOIN fx_rate f 
        ON to_date(t.tx_ts) = f.fx_date 
        AND t.currency = f.currency
""")

final_good_enriched.show(5)

+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+-----------+----------+
| mcc|       tx_id|              tx_ts|   card_no|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|home_country|is_foreign|      category|risk_weight|rate_to_azn|amount_azn|
+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+-----------+----------+
|4829|TX0008465429|2026-02-20 02:32:00|CARD300116|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:40:...| CUST100090|         AZE|      true|Money Transfer|          8|       1.85|    2001.7|
|5732|TX0010439369|2026-02-20 21:23:00|CARD300058|   127|     GBP|    AZE|Sumqayit|       NULL|ecom|      NULL|2026-02-24 06

In [173]:
final_good_enriched.createOrReplaceTempView("final_good_enriched")

final_good_enriched = spark.sql("""
    SELECT 
        *,
        CASE WHEN country != home_country THEN true ELSE false END AS is_foreign,
        CASE WHEN risk_weight >= 8        THEN true ELSE false END AS is_high_risk
    FROM final_good_enriched
""")

final_good_enriched.show(5)

+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+-----------+----------+----------+------------+
| mcc|       tx_id|              tx_ts|   card_no|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|home_country|is_foreign|      category|risk_weight|rate_to_azn|amount_azn|is_foreign|is_high_risk|
+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+----------+--------------+-----------+-----------+----------+----------+------------+
|4829|TX0008465429|2026-02-20 02:32:00|CARD300116|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:40:...| CUST100090|         AZE|      true|Money Transfer|          8|       1.85|    2001.7|      true|        true|
|5732|TX0010439369|2026-02-2

In [176]:
final_good_enriched = final_good_enriched.drop("is_foreign", "is_high_risk")

final_good_enriched.createOrReplaceTempView("final_good_enriched")

final_good_enriched = spark.sql("""
    SELECT 
        *,
        CASE WHEN country != home_country THEN true ELSE false END AS is_foreign,
        CASE WHEN risk_weight >= 8        THEN true ELSE false END AS is_high_risk
    FROM final_good_enriched
""")

final_good_enriched.write.mode("overwrite").parquet(silver_path + "transactions_good_enriched")
final_good_enriched.show(5)

+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+--------------+-----------+-----------+----------+----------+------------+
| mcc|       tx_id|              tx_ts|   card_no|amount|currency|country|    city|terminal_id|type|event_time|         insert_time|customer_no|home_country|      category|risk_weight|rate_to_azn|amount_azn|is_foreign|is_high_risk|
+----+------------+-------------------+----------+------+--------+-------+--------+-----------+----+----------+--------------------+-----------+------------+--------------+-----------+-----------+----------+----------+------------+
|4829|TX0008465429|2026-02-20 02:32:00|CARD300116|  1082|     EUR|    USA|New York|       NULL|ecom|      NULL|2026-02-24 06:47:...| CUST100090|         AZE|Money Transfer|          8|       1.85|    2001.7|      true|        true|
|5732|TX0010439369|2026-02-20 21:23:00|CARD300058|   127|     GBP|    AZ

In [177]:
final_good_enriched.createOrReplaceTempView("silver")
df_gold = spark.sql("""
    SELECT
        customer_no,
        to_date(tx_ts)  AS tx_date,
        COUNT(tx_id) AS total_txn_count,
        ROUND(SUM(amount_azn), 2) AS total_spent_azn,
        ROUND(AVG(amount_azn), 2)  AS avg_amount_azn,
        SUM(CAST(is_foreign   AS INT))  AS foreign_txn_count,
        SUM(CAST(is_high_risk AS INT))  AS high_risk_txn_count,
        COUNT(DISTINCT country)  AS distinct_country_count,
        ROUND(AVG(risk_weight), 4)                        AS avg_risk_weight

    FROM silver
    GROUP BY customer_no, to_date(tx_ts)
""")

In [178]:
df_gold.createOrReplaceTempView("gold")

In [179]:
df_gold.show()

+-----------+----------+---------------+---------------+--------------+-----------------+-------------------+----------------------+---------------+
|customer_no|   tx_date|total_txn_count|total_spent_azn|avg_amount_azn|foreign_txn_count|high_risk_txn_count|distinct_country_count|avg_risk_weight|
+-----------+----------+---------------+---------------+--------------+-----------------+-------------------+----------------------+---------------+
| CUST100033|2026-02-20|             12|        5626.84|         468.9|               12|                  4|                     4|         5.9167|
| CUST100054|2026-02-20|              4|         1048.4|         262.1|                4|                  0|                     2|            5.5|
| CUST100032|2026-02-20|              2|           45.3|         22.65|                0|                  1|                     1|            6.5|
| CUST100036|2026-02-20|              4|        1708.45|        427.11|                4|                 

In [180]:
df_gold = spark.sql("""
    SELECT
        *,
        ROUND(foreign_txn_count * 2 + high_risk_txn_count * 3 + avg_risk_weight, 2) AS risk_score
    FROM gold
""")

df_gold.createOrReplaceTempView("gold")

df_gold = spark.sql("""
    SELECT
        *,
        CASE 
            WHEN risk_score >= 20 THEN 'HIGH'
            WHEN risk_score >= 10 THEN 'MEDIUM'
            ELSE                       'LOW'
        END AS risk_level
    FROM gold
""")

In [181]:
df_gold_out = df_gold.coalesce(4)
df_gold_out.write.mode("overwrite").partitionBy("tx_date").format("delta").save(gold_path + "customer_daily_risk")

In [182]:
df_gold_out.write \
    .format("jdbc") \
    .option("url",           jdbc_url) \
    .option("dbtable",       "public.customer_daily_risk") \
    .option("user",          jdbc_props["user"]) \
    .option("password",      jdbc_props["password"]) \
    .option("driver",        jdbc_props["driver"]) \
    .option("batchsize",     "5000") \
    .option("numPartitions", "4") \
    .mode("overwrite") \
    .save()


In [183]:
spark.read \
    .format("jdbc") \
    .option("url",      jdbc_url) \
    .option("dbtable",  "public.customer_daily_risk") \
    .option("user",     jdbc_props["user"]) \
    .option("password", jdbc_props["password"]) \
    .option("driver",   jdbc_props["driver"]) \
    .load() \
    .show(5)

+-----------+----------+---------------+---------------+--------------+-----------------+-------------------+----------------------+---------------+----------+----------+
|customer_no|   tx_date|total_txn_count|total_spent_azn|avg_amount_azn|foreign_txn_count|high_risk_txn_count|distinct_country_count|avg_risk_weight|risk_score|risk_level|
+-----------+----------+---------------+---------------+--------------+-----------------+-------------------+----------------------+---------------+----------+----------+
| CUST100033|2026-02-20|             12|        5626.84|         468.9|               12|                  4|                     4|         5.9167|     41.92|      HIGH|
| CUST100054|2026-02-20|              4|         1048.4|         262.1|                4|                  0|                     2|            5.5|      13.5|    MEDIUM|
| CUST100032|2026-02-20|              2|           45.3|         22.65|                0|                  1|                     1|            6